[link text](https://)### Hệ thống chấm điểm tín dụng phi tập trung

Trong ngân hàng truyền thống, dữ liệu tín dụng thường bị chia cắt. Nếu bạn vay tiền ở ngân hàng A, ngân hàng B có thể không biết lịch sử trả nợ của bạn trừ khi họ truy cập vào hệ thống tập trung (như CIC tại Việt Nam). Hệ thống Blockchain này cho phép các tổ chức tài chính đóng góp "điểm tin cậy" cho khách hàng mà không làm lộ chi tiết giao dịch cụ thể. Mục tiêu là Xây dựng một mạng lưới giữa các ngân hàng và tổ chức tài chính để ghi lại lịch sử tín dụng của khách hàng.

### Gợi ý Tính năng tài chính:
* Consensus Scoring: Điểm tín dụng được tổng hợp từ nhiều nguồn (Ngân hàng, Công ty Tài chính, Sàn TMĐT).
* Bất biến: Lịch sử nợ xấu không thể bị xóa hoặc sửa đổi bởi cá nhân.
* Proof of Credit: Khách hàng có thể chứng minh mình có "điểm tốt" để hưởng lãi suất thấp mà không cần in sao kê giấy




### Khắc phục lỗi và minh họa tính năng bảo mật

In [13]:
import hashlib
import json
import time
from collections import defaultdict

class CreditRecord:
    """Đại diện cho một bản ghi tín dụng duy nhất từ một tổ chức tài chính.
    Có thể tham chiếu đến một bản ghi khác nếu đây là bản ghi đính chính."""
    def __init__(self, user_id, financial_institution_id, score, timestamp=None, correction_for_record_hash=None):
        self.user_id = user_id
        self.financial_institution_id = financial_institution_id
        self.score = score  # Ví dụ: điểm từ 0 đến 100
        self.timestamp = timestamp if timestamp else time.time()
        self.correction_for_record_hash = correction_for_record_hash # Hash của bản ghi cũ mà bản ghi này đính chính

    def to_dict(self):
        return {
            'user_id': self.user_id,
            'financial_institution_id': self.financial_institution_id,
            'score': self.score,
            'timestamp': self.timestamp,
            'correction_for_record_hash': self.correction_for_record_hash
        }

    def compute_hash(self):
        """Tính hash của bản ghi để có thể tham chiếu đến nó."""
        record_string = json.dumps(self.to_dict(), sort_keys=True)
        return hashlib.sha256(record_string.encode()).hexdigest()

    def __repr__(self):
        corr_info = f", Corrects: {self.correction_for_record_hash[:6]}..." if self.correction_for_record_hash else ""
        return f"CreditRecord(User: {self.user_id}, FI: {self.financial_institution_id}, Score: {self.score}, Time: {self.timestamp:.0f}{corr_info})"

class Block:
    """Đại diện cho một khối trong chuỗi khối, chứa các bản ghi tín dụng."""
    def __init__(self, index, previous_hash, records, timestamp=None, nonce=0):
        self.index = index
        self.timestamp = timestamp if timestamp else time.time()
        self.records = records  # Danh sách các đối tượng CreditRecord
        self.previous_hash = previous_hash
        self.nonce = nonce # Dùng cho Proof-of-Work (đơn giản)

    def compute_hash(self):
        block_string = json.dumps(self.to_dict_for_hash(), sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

    def to_dict_for_hash(self):
        return {
            'index': self.index,
            'timestamp': self.timestamp,
            'records': [record.to_dict() for record in self.records],
            'previous_hash': self.previous_hash,
            'nonce': self.nonce
        }

class DecentralizedCreditSystem:
    """Mô phỏng chuỗi khối quản lý các bản ghi tín dụng."""
    def __init__(self):
        self.chain = []
        self.pending_records = []
        self.create_genesis_block()

    def create_genesis_block(self):
        """Tạo khối khởi tạo (khối đầu tiên) của chuỗi."""
        genesis_block = Block(0, "0", [])
        self.chain.append(genesis_block)

    @property
    def last_block(self):
        """Trả về khối cuối cùng trong chuỗi."""
        return self.chain[-1]

    def add_record(self, record):
        """Thêm một bản ghi tín dụng vào danh sách chờ xử lý."""
        self.pending_records.append(record)
        print(f"Thêm bản ghi mới vào danh sách chờ: {record}")

    def add_correction_record(self, user_id, financial_institution_id, new_score, corrected_record_hash):
        """Thêm một bản ghi đính chính, tham chiếu đến bản ghi bị lỗi."""
        correction = CreditRecord(
            user_id=user_id,
            financial_institution_id=financial_institution_id,
            score=new_score,
            correction_for_record_hash=corrected_record_hash
        )
        self.add_record(correction)
        print(f"Thêm bản ghi đính chính cho {user_id} từ {financial_institution_id} (điểm mới: {new_score}) cho bản ghi hash {corrected_record_hash[:6]}...")

    def mine_block(self):
        """Tạo một khối mới từ các bản ghi đang chờ xử lý."""
        if not self.pending_records:
            print("Không có bản ghi nào đang chờ để tạo khối mới.")
            return None

        last_block = self.last_block
        new_block_index = last_block.index + 1
        previous_hash = last_block.compute_hash()

        # Lấy tất cả các bản ghi đang chờ và xóa chúng
        records_to_add = list(self.pending_records)
        self.pending_records = []

        new_block = Block(new_block_index, previous_hash, records_to_add)
        # Để đơn giản, bỏ qua Proof-of-Work phức tạp
        self.chain.append(new_block)
        print(f"Một khối mới đã được tạo và thêm vào chuỗi: Index {new_block.index}, Hash: {new_block.compute_hash()}")
        return new_block

    def get_user_credit_history(self, user_id):
        """Lấy tất cả các bản ghi tín dụng cho một người dùng cụ thể."""
        history = []
        for block in self.chain:
            for record in block.records:
                if record.user_id == user_id:
                    history.append(record)
        return history

    def calculate_consensus_score(self, user_id):
        """Tính điểm tín dụng đồng thuận dựa trên tất cả các bản ghi hiện có,
        ưu tiên bản ghi đính chính nếu có."""
        user_history = self.get_user_credit_history(user_id)
        if not user_history:
            return "Không có dữ liệu tín dụng cho người dùng này."

        # Sắp xếp lịch sử theo thời gian để đảm bảo xử lý các bản đính chính đúng cách
        user_history.sort(key=lambda r: r.timestamp)

        # Lưu trữ bản ghi hợp lệ cuối cùng cho mỗi tổ chức tài chính
        current_valid_records = {}
        # Lưu trữ các bản ghi đã được đính chính (đã bị ghi đè)
        superseded_records = set()

        for record in user_history:
            # Nếu bản ghi này là một bản đính chính, đánh dấu bản ghi gốc là đã bị ghi đè
            if record.correction_for_record_hash:
                superseded_records.add(record.correction_for_record_hash)

            # Thêm bản ghi hiện tại vào danh sách các bản ghi hợp lệ
            # Nếu có một bản đính chính cho cùng một FI và người dùng, bản ghi này sẽ được ưu tiên
            current_valid_records[(record.user_id, record.financial_institution_id)] = record

        final_scores = []
        for (u_id, fi_id), record in current_valid_records.items():
            # Chỉ sử dụng các bản ghi không bị đính chính bởi một bản ghi khác
            if record.compute_hash() not in superseded_records:
                final_scores.append(record.score)

        if not final_scores:
            return "Không có điểm hợp lệ để tính toán sau khi xử lý đính chính."

        # Điểm đồng thuận là trung bình của các điểm hợp lệ cuối cùng từ mỗi FI
        consensus_score = sum(final_scores) / len(final_scores)
        return consensus_score

    def is_chain_valid(self):
        """Kiểm tra tính hợp lệ của chuỗi khối (kiểm tra hash và liên kết)."""
        for i in range(1, len(self.chain)):
            current_block = self.chain[i]
            previous_block = self.chain[i-1]

            # Kiểm tra hash của khối hiện tại
            # Cần tính toán lại hash để đảm bảo tính toàn vẹn
            if current_block.compute_hash() != self.chain[i].compute_hash():
                print(f"Lỗi: Hash khối hiện tại {current_block.index} không khớp.")
                return False

            # Kiểm tra xem previous_hash có khớp không
            if current_block.previous_hash != previous_block.compute_hash():
                print(f"Lỗi: previous_hash của khối {current_block.index} không khớp với hash của khối trước đó.")
                return False
        return True

    def tamper_with_block(self, block_index, new_records):
        """Thay đổi các bản ghi trong một khối cụ thể để minh họa việc giả mạo dữ liệu.
        Thao tác này sẽ làm mất hiệu lực của chuỗi."""
        if block_index < 0 or block_index >= len(self.chain):
            print("Chỉ số khối không hợp lệ.")
            return

        print(f"Giả mạo khối tại chỉ số: {block_index}...")
        tampered_block = self.chain[block_index]
        tampered_block.records = new_records
        # Sau khi thay đổi records, hash của khối sẽ thay đổi, làm cho chuỗi không hợp lệ
        # Chúng ta không cần cập nhật hash ở đây vì is_chain_valid sẽ tính toán lại và so sánh.
        print(f"Đã thay đổi dữ liệu trong khối {block_index}. Hash của khối này sẽ thay đổi.")

### Minh họa Cơ chế Xử lý Khiếu nại (Correction Record)

In [14]:
# Khởi tạo lại hệ thống để có một chuỗi khối mới cho ví dụ đính chính
credit_system_correction = DecentralizedCreditSystem()

print("--- Ví dụ bản ghi đính chính ---")

# Bước 1: Ngân hàng A gửi một bản ghi lỗi cho John (điểm 20)
record_r001_john_bank_a_error = CreditRecord(user_id=user_john, financial_institution_id=fi_bank_a, score=20)
credit_system_correction.add_record(record_r001_john_bank_a_error)

# Tạo khối đầu tiên chứa bản ghi lỗi
print("\n--- Tạo khối 1 với bản ghi lỗi ---")
credit_system_correction.mine_block()

# Lấy hash của bản ghi lỗi để tham chiếu
hash_r001 = record_r001_john_bank_a_error.compute_hash()
print(f"Hash của bản ghi lỗi (R001): {hash_r001[:10]}...")

# Bước 2: Công ty Tài chính B gửi bản ghi thông thường cho John (điểm 75)
credit_system_correction.add_record(CreditRecord(user_id=user_john, financial_institution_id=fi_finance_co_b, score=75))

# Bước 3: Ngân hàng A tạo bản ghi đính chính cho John, sửa điểm thành 80 và tham chiếu R001
credit_system_correction.add_correction_record(
    user_id=user_john,
    financial_institution_id=fi_bank_a,
    new_score=80,
    corrected_record_hash=hash_r001
)

# Tạo khối thứ hai chứa bản ghi thường và bản ghi đính chính
print("\n--- Tạo khối 2 với bản ghi thường và bản ghi đính chính ---")
credit_system_correction.mine_block()

print("\n--- Lịch sử tín dụng của John (sau đính chính) ---")
john_history_correction = credit_system_correction.get_user_credit_history(user_john)
for record in john_history_correction:
    print(record)

print("\n--- Điểm tín dụng đồng thuận của John (sau đính chính) ---")
john_score_after_correction = credit_system_correction.calculate_consensus_score(user_john)
print(f"Điểm tín dụng đồng thuận của John: {john_score_after_correction:.2f}")

print("\n--- Kiểm tra tính hợp lệ của chuỗi sau đính chính ---")
print(f"Chuỗi khối hợp lệ: {credit_system_correction.is_chain_valid()}")

# Ví dụ để xác minh rằng điểm lỗi (20) không được tính vào điểm đồng thuận,
# mà điểm đính chính (80) được sử dụng.

--- Ví dụ bản ghi đính chính ---
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-001-JohnDoe, FI: Ngân hàng A, Score: 20, Time: 1781729633)

--- Tạo khối 1 với bản ghi lỗi ---
Một khối mới đã được tạo và thêm vào chuỗi: Index 1, Hash: 958e5dae58127bd8c09ecf773545c09df4bc68f913f5d0b3274793dc9e4db8dd
Hash của bản ghi lỗi (R001): 060bb703dd...
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-001-JohnDoe, FI: Công ty Tài chính B, Score: 75, Time: 1781729633)
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-001-JohnDoe, FI: Ngân hàng A, Score: 80, Time: 1781729633, Corrects: 060bb7...)
Thêm bản ghi đính chính cho USER-001-JohnDoe từ Ngân hàng A (điểm mới: 80) cho bản ghi hash 060bb7...

--- Tạo khối 2 với bản ghi thường và bản ghi đính chính ---
Một khối mới đã được tạo và thêm vào chuỗi: Index 2, Hash: 3e9c74cb74b222c8c6306ce8c6469bbb2c781553dcb66b79374b05e0b9dc0bac

--- Lịch sử tín dụng của John (sau đính chính) ---
CreditRecord(User: USER-001-JohnDoe, FI

In [12]:
# Khởi tạo lại hệ thống tín dụng phi tập trung
credit_system = DecentralizedCreditSystem()

# Giả định các tổ chức tài chính và người dùng
fi_bank_a = "Ngân hàng A"
fi_finance_co_b = "Công ty Tài chính B"
fi_e_commerce_c = "Sàn TMĐT C"

user_john = "USER-001-JohnDoe"
user_jane = "USER-002-JaneSmith"

print("--- Thêm các bản ghi tín dụng ---")
# Ngân hàng A cấp điểm cho John
credit_system.add_record(CreditRecord(user_john, fi_bank_a, 85))
# Công ty Tài chính B cấp điểm cho John
credit_system.add_record(CreditRecord(user_john, fi_finance_co_b, 70))
# Sàn TMĐT C cấp điểm cho John
credit_system.add_record(CreditRecord(user_john, fi_e_commerce_c, 90))

# Jane cũng nhận được điểm
credit_system.add_record(CreditRecord(user_jane, fi_bank_a, 75))
credit_system.add_record(CreditRecord(user_jane, fi_finance_co_b, 80))

# Tạo một khối mới để ghi lại các bản ghi này
print("\n--- Tạo khối đầu tiên ---")
credit_system.mine_block()

# Thêm thêm một vài bản ghi
print("\n--- Thêm thêm bản ghi ---")
credit_system.add_record(CreditRecord(user_john, fi_bank_a, 88)) # John nhận điểm mới từ Ngân hàng A
credit_system.add_record(CreditRecord(user_jane, fi_e_commerce_c, 65)) # Jane nhận điểm từ Sàn TMĐT C

# Tạo khối thứ hai
print("\n--- Tạo khối thứ hai ---")
credit_system.mine_block()

print("\n--- Lịch sử tín dụng của John ---")
john_history = credit_system.get_user_credit_history(user_john)
for record in john_history:
    print(record)

print("\n--- Điểm tín dụng đồng thuận của John ---")
john_score = credit_system.calculate_consensus_score(user_john)
print(f"Điểm tín dụng đồng thuận của John: {john_score:.2f}")

print("\n--- Lịch sử tín dụng của Jane ---")
jane_history = credit_system.get_user_credit_history(user_jane)
for record in jane_history:
    print(record)

print("\n--- Điểm tín dụng đồng thuận của Jane ---")
jane_score = credit_system.calculate_consensus_score(user_jane)
print(f"Điểm tín dụng đồng thuận của Jane: {jane_score:.2f}")

print("\n--- Kiểm tra tính hợp lệ của chuỗi ban đầu ---")
print(f"Chuỗi khối hợp lệ: {credit_system.is_chain_valid()}")

--- Thêm các bản ghi tín dụng ---
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-001-JohnDoe, FI: Ngân hàng A, Score: 85, Time: 1781727820)
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-001-JohnDoe, FI: Công ty Tài chính B, Score: 70, Time: 1781727820)
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-001-JohnDoe, FI: Sàn TMĐT C, Score: 90, Time: 1781727820)
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-002-JaneSmith, FI: Ngân hàng A, Score: 75, Time: 1781727820)
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-002-JaneSmith, FI: Công ty Tài chính B, Score: 80, Time: 1781727820)

--- Tạo khối đầu tiên ---
Một khối mới đã được tạo và thêm vào chuỗi: Index 1, Hash: ee48203f66604795fd8d7f29d9a798800046d1ac78d89bf3353e26531508c8e3

--- Thêm thêm bản ghi ---
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-001-JohnDoe, FI: Ngân hàng A, Score: 88, Time: 1781727820)
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: 

### Minh họa giả mạo dữ liệu và kiểm tra tính hợp lệ

In [7]:
print("\n--- Cố gắng giả mạo dữ liệu trong khối 1 (khối thứ hai trong chuỗi) ---")
# Tạo một bản ghi mới giả mạo để thay thế
tampered_record = CreditRecord(user_john, fi_bank_a, 10, timestamp=time.time()) # Thay đổi điểm của John thành 10

# Khối genesis là 0, khối đầu tiên được mine là 1, khối thứ hai được mine là 2.
# Giả mạo khối tại index 1
credit_system.tamper_with_block(1, [tampered_record])

print("\n--- Kiểm tra lại tính hợp lệ của chuỗi sau khi giả mạo ---")
print(f"Chuỗi khối hợp lệ: {credit_system.is_chain_valid()}")

print("\n--- Lịch sử tín dụng của John sau khi giả mạo ---")
john_history_after_tamper = credit_system.get_user_credit_history(user_john)
for record in john_history_after_tamper:
    print(record)

print("\n--- Điểm tín dụng đồng thuận của John sau khi giả mạo ---")
john_score_after_tamper = credit_system.calculate_consensus_score(user_john)
print(f"Điểm tín dụng đồng thuận của John: {john_score_after_tamper:.2f}")


--- Cố gắng giả mạo dữ liệu trong khối 1 (khối thứ hai trong chuỗi) ---
Giả mạo khối tại chỉ số: 1...
Đã thay đổi dữ liệu trong khối 1. Hash của khối này sẽ thay đổi.

--- Kiểm tra lại tính hợp lệ của chuỗi sau khi giả mạo ---
Lỗi: previous_hash của khối 2 không khớp với hash của khối trước đó.
Chuỗi khối hợp lệ: False

--- Lịch sử tín dụng của John sau khi giả mạo ---
CreditRecord(User: USER-001-JohnDoe, FI: Ngân hàng A, Score: 10, Time: 1781727752)
CreditRecord(User: USER-001-JohnDoe, FI: Ngân hàng A, Score: 88, Time: 1781727746)

--- Điểm tín dụng đồng thuận của John sau khi giả mạo ---
Điểm tín dụng đồng thuận của John: 49.00


### Mô phỏng Hệ thống Chấm điểm Tín dụng Phi tập trung

Đây là một mô hình Python đơn giản để minh họa các nguyên tắc cơ bản của hệ thống chấm điểm tín dụng phi tập trung, nơi các tổ chức tài chính có thể đóng góp điểm và lịch sử này là bất biến.

In [8]:
import hashlib
import json
import time
from collections import defaultdict

class CreditRecord:
    """Đại diện cho một bản ghi tín dụng duy nhất từ một tổ chức tài chính."""
    def __init__(self, user_id, financial_institution_id, score, timestamp=None):
        self.user_id = user_id
        self.financial_institution_id = financial_institution_id
        self.score = score  # Ví dụ: điểm từ 0 đến 100
        self.timestamp = timestamp if timestamp else time.time()

    def to_dict(self):
        return {
            'user_id': self.user_id,
            'financial_institution_id': self.financial_institution_id,
            'score': self.score,
            'timestamp': self.timestamp
        }

    def __repr__(self):
        return f"CreditRecord(User: {self.user_id}, FI: {self.financial_institution_id}, Score: {self.score}, Time: {self.timestamp:.0f})"

In [9]:
class Block:
    """Đại diện cho một khối trong chuỗi khối, chứa các bản ghi tín dụng."""
    def __init__(self, index, previous_hash, records, timestamp=None, nonce=0):
        self.index = index
        self.timestamp = timestamp if timestamp else time.time()
        self.records = records  # Danh sách các đối tượng CreditRecord
        self.previous_hash = previous_hash
        self.nonce = nonce # Dùng cho Proof-of-Work (đơn giản)

    def compute_hash(self):
        block_string = json.dumps(self.to_dict_for_hash(), sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

    def to_dict_for_hash(self):
        return {
            'index': self.index,
            'timestamp': self.timestamp,
            'records': [record.to_dict() for record in self.records],
            'previous_hash': self.previous_hash,
            'nonce': self.nonce
        }

In [10]:
class DecentralizedCreditSystem:
    """Mô phỏng chuỗi khối quản lý các bản ghi tín dụng."""
    def __init__(self):
        self.chain = []
        self.pending_records = []
        self.create_genesis_block()

    def create_genesis_block(self):
        """Tạo khối khởi tạo (khối đầu tiên) của chuỗi."""
        genesis_block = Block(0, "0", [])
        self.chain.append(genesis_block)

    @property
    def last_block(self):
        """Trả về khối cuối cùng trong chuỗi."""
        return self.chain[-1]

    def add_record(self, record):
        """Thêm một bản ghi tín dụng vào danh sách chờ xử lý."""
        self.pending_records.append(record)
        print(f"Thêm bản ghi mới vào danh sách chờ: {record}")

    def mine_block(self):
        """Tạo một khối mới từ các bản ghi đang chờ xử lý."""
        if not self.pending_records:
            print("Không có bản ghi nào đang chờ để tạo khối mới.")
            return None

        last_block = self.last_block
        new_block_index = last_block.index + 1
        previous_hash = last_block.compute_hash()

        # Lấy tất cả các bản ghi đang chờ và xóa chúng
        records_to_add = list(self.pending_records)
        self.pending_records = []

        new_block = Block(new_block_index, previous_hash, records_to_add)
        # Để đơn giản, bỏ qua Proof-of-Work phức tạp
        self.chain.append(new_block)
        print(f"Một khối mới đã được tạo và thêm vào chuỗi: Index {new_block.index}, Hash: {new_block.compute_hash()}")
        return new_block

    def get_user_credit_history(self, user_id):
        """Lấy tất cả các bản ghi tín dụng cho một người dùng cụ thể."""
        history = []
        for block in self.chain:
            for record in block.records:
                if record.user_id == user_id:
                    history.append(record)
        return history

    def calculate_consensus_score(self, user_id):
        """Tính điểm tín dụng đồng thuận dựa trên tất cả các bản ghi hiện có."""
        user_history = self.get_user_credit_history(user_id)
        if not user_history:
            return "Không có dữ liệu tín dụng cho người dùng này."

        # Ví dụ đơn giản: tính điểm trung bình từ tất cả các tổ chức
        scores_by_fi = defaultdict(list)
        for record in user_history:
            scores_by_fi[record.financial_institution_id].append(record.score)

        final_scores = []
        for fi_id, scores in scores_by_fi.items():
            # Có thể có logic phức tạp hơn ở đây, ví dụ: lấy điểm gần nhất, điểm cao nhất, v.v.
            # Hiện tại, lấy điểm trung bình của từng FI
            final_scores.append(sum(scores) / len(scores))

        if not final_scores:
            return "Không có điểm hợp lệ để tính toán."

        # Điểm đồng thuận là trung bình của các điểm trung bình từ mỗi FI
        consensus_score = sum(final_scores) / len(final_scores)
        return consensus_score

    def is_chain_valid(self):
        """Kiểm tra tính hợp lệ của chuỗi khối (kiểm tra hash và liên kết)."""
        for i in range(1, len(self.chain)):
            current_block = self.chain[i]
            previous_block = self.chain[i-1]

            # Kiểm tra hash của khối hiện tại
            # Cần tính toán lại hash để đảm bảo tính toàn vẹn
            if current_block.compute_hash() != self.chain[i].compute_hash():
                print(f"Lỗi: Hash khối hiện tại {current_block.index} không khớp.")
                return False

            # Kiểm tra xem previous_hash có khớp không
            if current_block.previous_hash != previous_block.compute_hash():
                print(f"Lỗi: previous_hash của khối {current_block.index} không khớp với hash của khối trước đó.")
                return False
        return True

    def tamper_with_block(self, block_index, new_records):
        """Thay đổi các bản ghi trong một khối cụ thể để minh họa việc giả mạo dữ liệu.
        Thao tác này sẽ làm mất hiệu lực của chuỗi."""
        if block_index < 0 or block_index >= len(self.chain):
            print("Chỉ số khối không hợp lệ.")
            return

        print(f"Giả mạo khối tại chỉ số: {block_index}...")
        tampered_block = self.chain[block_index]
        tampered_block.records = new_records
        # Sau khi thay đổi records, hash của khối sẽ thay đổi, làm cho chuỗi không hợp lệ
        # Chúng ta không cần cập nhật hash ở đây vì is_chain_valid sẽ tính toán lại và so sánh.
        print(f"Đã thay đổi dữ liệu trong khối {block_index}. Hash của khối này sẽ thay đổi.")


### Minh họa Cách sử dụng Hệ thống

In [11]:
# Khởi tạo hệ thống tín dụng phi tập trung
credit_system = DecentralizedCreditSystem()

# Giả định các tổ chức tài chính và người dùng
fi_bank_a = "Ngân hàng A"
fi_finance_co_b = "Công ty Tài chính B"
fi_e_commerce_c = "Sàn TMĐT C"

user_john = "USER-001-JohnDoe"
user_jane = "USER-002-JaneSmith"

print("--- Thêm các bản ghi tín dụng ---")
# Ngân hàng A cấp điểm cho John
credit_system.add_record(CreditRecord(user_john, fi_bank_a, 85))
# Công ty Tài chính B cấp điểm cho John
credit_system.add_record(CreditRecord(user_john, fi_finance_co_b, 70))
# Sàn TMĐT C cấp điểm cho John
credit_system.add_record(CreditRecord(user_john, fi_e_commerce_c, 90))

# Jane cũng nhận được điểm
credit_system.add_record(CreditRecord(user_jane, fi_bank_a, 75))
credit_system.add_record(CreditRecord(user_jane, fi_finance_co_b, 80))

# Tạo một khối mới để ghi lại các bản ghi này
print("\n--- Tạo khối đầu tiên ---")
credit_system.mine_block()

# Thêm thêm một vài bản ghi
print("\n--- Thêm thêm bản ghi ---")
credit_system.add_record(CreditRecord(user_john, fi_bank_a, 88)) # John nhận điểm mới từ Ngân hàng A
credit_system.add_record(CreditRecord(user_jane, fi_e_commerce_c, 65)) # Jane nhận điểm từ Sàn TMĐT C

# Tạo khối thứ hai
print("\n--- Tạo khối thứ hai ---")
credit_system.mine_block()

print("\n--- Lịch sử tín dụng của John ---")
john_history = credit_system.get_user_credit_history(user_john)
for record in john_history:
    print(record)

print("\n--- Điểm tín dụng đồng thuận của John ---")
john_score = credit_system.calculate_consensus_score(user_john)
print(f"Điểm tín dụng đồng thuận của John: {john_score:.2f}")

print("\n--- Lịch sử tín dụng của Jane ---")
jane_history = credit_system.get_user_credit_history(user_jane)
for record in jane_history:
    print(record)

print("\n--- Điểm tín dụng đồng thuận của Jane ---")
jane_score = credit_system.calculate_consensus_score(user_jane)
print(f"Điểm tín dụng đồng thuận của Jane: {jane_score:.2f}")

print("\n--- Kiểm tra tính hợp lệ của chuỗi ---")
print(f"Chuỗi khối hợp lệ: {credit_system.is_chain_valid()}")

--- Thêm các bản ghi tín dụng ---
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-001-JohnDoe, FI: Ngân hàng A, Score: 85, Time: 1781727763)
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-001-JohnDoe, FI: Công ty Tài chính B, Score: 70, Time: 1781727763)
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-001-JohnDoe, FI: Sàn TMĐT C, Score: 90, Time: 1781727763)
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-002-JaneSmith, FI: Ngân hàng A, Score: 75, Time: 1781727763)
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-002-JaneSmith, FI: Công ty Tài chính B, Score: 80, Time: 1781727763)

--- Tạo khối đầu tiên ---
Một khối mới đã được tạo và thêm vào chuỗi: Index 1, Hash: 89b8936d02cda8511ba37e39cbb1cb924a5e88c38ca1327a32180cf15a737e26

--- Thêm thêm bản ghi ---
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: USER-001-JohnDoe, FI: Ngân hàng A, Score: 88, Time: 1781727763)
Thêm bản ghi mới vào danh sách chờ: CreditRecord(User: 